# 01 — YASA Sleep Staging Demo

**Purpose:** Demonstrate automated sleep staging using YASA.
YASA's pre-trained LightGBM achieves ~85% accuracy, matching human scorers.

**Stages:** Wake (W), N1, N2, N3 (deep/slow-wave), REM

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
import yasa
import mne
plt.style.use('dark_background')

from classifiers.tmr.sleep_staging import SleepStagingConfig, run_sleep_staging

## 1. Load Sample Sleep EEG

Using YASA's built-in sample data for demo.

In [ ]:
# Load YASA sample night of sleep
import yasa
# For demo, create synthetic sleep-like data
# Replace with real sleep EEG when available
sfreq = 100
duration_hrs = 0.1  # 6 min demo
n_samples = int(sfreq * duration_hrs * 3600)
times = np.arange(n_samples) / sfreq

# Simulate: slow waves (delta) + spindles (12-15 Hz)
rng = np.random.RandomState(42)
delta = 50e-6 * np.sin(2 * np.pi * 1.0 * times)
spindles = 10e-6 * np.sin(2 * np.pi * 13 * times) * (rng.random(n_samples) > 0.95)
noise = 5e-6 * rng.randn(n_samples)
eeg = delta + spindles + noise

info = mne.create_info(['C4'], sfreq=sfreq, ch_types='eeg')
raw = mne.io.RawArray(eeg[np.newaxis, :], info, verbose=False)
print(f'Duration: {raw.times[-1]/60:.1f} minutes')

## 2. Run Sleep Staging

In [ ]:
# Run YASA staging
try:
    result = run_sleep_staging(raw, SleepStagingConfig(eeg_name='C4'))
    print(f'Stages: {result.stage_durations_min}')
    print(f'Sleep efficiency: {result.sleep_efficiency:.1f}%')
    print(f'N2/N3 epochs (TMR targets): {len(result.n2_n3_epochs)}')
except Exception as e:
    print(f'Staging requires longer recording: {e}')
    print('Use real sleep data (6+ hours) for proper staging')